In [1]:
import json
import gzip
import glob
import re
from collections import Counter
import os

INPUT_FILES = glob.glob("/home/stefano-u/fuzzing_lab/shared_corpus/dataset_syzkaller_*.jsonl.gz")
REPORT_FILE = "/home/stefano-u/fuzzing_lab/shared_corpus/report_errori.txt"

def estrai_vero_errore(verifier_log):
    linee = verifier_log.strip().split('\n')
    for linea in reversed(linee):
        linea = linea.strip()
        if not linea: continue
        if linea.startswith("processed ") and "insns" in linea: continue
        if "R0=" in linea or "R1=" in linea or linea.startswith("mark_precise:"): continue
        return linea
    return "unknown_error"

def normalizza_errore(errore_raw):
    err = str(errore_raw)
    err = re.sub(r'[-+]?\b\d+\b', '<NUM>', err)
    err = re.sub(r'0x[0-9a-fA-F]+', '<HEX>', err)
    return err.strip()

def profila_dataset():
    print(f"[*] Avvio analisi su {len(INPUT_FILES)} file compressi...")
    
    contatore_errori = Counter()
    statistiche = {
        "totali_letti": 0,
        "json_malformati": 0,
        "senza_bytecode": 0,
        "programmi_validi": 0,
        "programmi_invalidi": 0
    }
    
    for file_path in INPUT_FILES:
        print(f"    -> Scansiono: {os.path.basename(file_path)}")
        try:
            with gzip.open(file_path, 'rt', encoding='utf-8') as f:
                for linea in f:
                    statistiche["totali_letti"] += 1
                    
                    # 1. Filtro JSON malformati
                    try:
                        sample = json.loads(linea)
                    except json.JSONDecodeError:
                        statistiche["json_malformati"] += 1
                        continue
                        
                    # 2. Filtro entry senza codice
                    bytecode = sample.get("bytecode_hex", "")
                    if not bytecode:
                        statistiche["senza_bytecode"] += 1
                        continue
                        
                    # 3. Conteggio validi/invalidi
                    is_valid = sample.get("is_valid", False)
                    if is_valid:
                        statistiche["programmi_validi"] += 1
                        continue
                    
                    statistiche["programmi_invalidi"] += 1
                    
                    # 4. Estrazione e conteggio classi di errore
                    vero_errore = estrai_vero_errore(sample.get("verifier_log", ""))
                    classe_errore = normalizza_errore(vero_errore)
                    contatore_errori[classe_errore] += 1
                    
        except Exception as e:
            print(f"       [!] Errore critico nel leggere {file_path}: {e}")
            continue

    # --- SALVATAGGIO REPORT ---
    print("\n[*] Scrittura del report in corso...")
    with open(REPORT_FILE, 'w', encoding='utf-8') as f:
        f.write("=== STATISTICHE GLOBALI DATASET ===\n")
        f.write(f"Righe totali processate: {statistiche['totali_letti']}\n")
        f.write(f"JSON scartati (malformati): {statistiche['json_malformati']}\n")
        f.write(f"Entry scartate (senza bytecode): {statistiche['senza_bytecode']}\n")
        f.write(f"Programmi VALIDI (Success): {statistiche['programmi_validi']}\n")
        f.write(f"Programmi INVALIDI (Failed): {statistiche['programmi_invalidi']}\n\n")
        
        f.write("=== DISTRIBUZIONE CLASSI DI ERRORE ===\n")
        f.write(f"Numero totale di CLASSI UNICHE trovate: {len(contatore_errori)}\n\n")
        
        # Ordina dal più frequente al meno frequente
        for errore, conteggio in contatore_errori.most_common():
            f.write(f"[{conteggio} programmi] -> {errore}\n")

    print(f"[+] Finito! Apri il file {REPORT_FILE} per vedere i risultati.")

profila_dataset()

[*] Avvio analisi su 73 file compressi...
    -> Scansiono: dataset_syzkaller_353_20260409_0600.jsonl.gz
    -> Scansiono: dataset_syzkaller_353_20260328_0145.jsonl.gz
    -> Scansiono: dataset_syzkaller_355_20260409_0145.jsonl.gz
    -> Scansiono: dataset_syzkaller_356_20260409_0515.jsonl.gz
    -> Scansiono: dataset_syzkaller_349_20260409_0630.jsonl.gz
    -> Scansiono: dataset_syzkaller_355_20260408_2215.jsonl.gz
    -> Scansiono: dataset_syzkaller_354_20260328_0315.jsonl.gz
    -> Scansiono: dataset_syzkaller_355_20260409_0715.jsonl.gz
    -> Scansiono: dataset_syzkaller_353_20260327_2330.jsonl.gz
    -> Scansiono: dataset_syzkaller_355_20260328_0345.jsonl.gz
    -> Scansiono: dataset_syzkaller_350_20260408_2315.jsonl.gz
    -> Scansiono: dataset_syzkaller_353_20260409_0545.jsonl.gz
    -> Scansiono: dataset_syzkaller_353_20260328_0000.jsonl.gz
    -> Scansiono: dataset_syzkaller_352_20260408_2345.jsonl.gz
    -> Scansiono: dataset_syzkaller_354_20260327_2345.jsonl.gz
    -> Scansi

In [2]:
import json
import gzip
import glob
import re
import os
from collections import defaultdict

INPUT_FILES = glob.glob("/home/stefano-u/fuzzing_lab/shared_corpus/dataset_syzkaller_*.jsonl.gz")
OUTPUT_FILE = "/home/stefano-u/fuzzing_lab/shared_corpus/dataset_final_qwen.jsonl"

# Parametri di bilanciamento decisi in base al report
MAX_INVALID_PER_CLASS = 2000 

def estrai_vero_errore(verifier_log):
    linee = verifier_log.strip().split('\n')
    for linea in reversed(linee):
        linea = linea.strip()
        if not linea: continue
        if linea.startswith("processed ") and "insns" in linea: continue
        if "R0=" in linea or "R1=" in linea or linea.startswith("mark_precise:"): continue
        return linea
    return "unknown_error"

def normalizza_errore(errore_raw):
    err = str(errore_raw)
    err = re.sub(r'[-+]?\b\d+\b', '<NUM>', err)
    err = re.sub(r'0x[0-9a-fA-F]+', '<HEX>', err)
    return err.strip()

def genera_dataset():
    print(f"[*] Generazione dataset finale in corso...")
    
    conteggio_classi = defaultdict(int)
    tot_validi = 0
    tot_invalidi = 0

    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f_out:
        for file_path in INPUT_FILES:
            print(f"    -> Elaborazione: {os.path.basename(file_path)}")
            with gzip.open(file_path, 'rt', encoding='utf-8') as f_in:
                for linea in f_in:
                    try:
                        sample = json.loads(linea)
                    except: continue

                    bytecode = sample.get("bytecode_hex", "")
                    if not bytecode: continue

                    is_valid = sample.get("is_valid", False)

                    if is_valid:
                        # TENIAMO TUTTI I VALIDI
                        f_out.write(json.dumps(sample) + "\n")
                        tot_validi += 1
                    else:
                        # FILTRIAMO GLI INVALIDI PER CLASSE
                        vero_errore = estrai_vero_errore(sample.get("verifier_log", ""))
                        classe = normalizza_errore(vero_errore)
                        
                        if conteggio_classi[classe] < MAX_INVALID_PER_CLASS:
                            # Aggiorniamo il sample con il vero errore estratto prima di salvarlo
                            sample["error_reason_clean"] = vero_errore
                            f_out.write(json.dumps(sample) + "\n")
                            conteggio_classi[classe] += 1
                            tot_invalidi += 1

    print(f"\n[!] Dataset completato!")
    print(f"    - Programmi VALIDI salvati: {tot_validi}")
    print(f"    - Programmi INVALIDI salvati: {tot_invalidi}")
    print(f"    - Totale entry per il training: {tot_validi + tot_invalidi}")
    print(f"    - File pronto in: {OUTPUT_FILE}")

genera_dataset()

[*] Generazione dataset finale in corso...
    -> Elaborazione: dataset_syzkaller_353_20260409_0600.jsonl.gz
    -> Elaborazione: dataset_syzkaller_353_20260328_0145.jsonl.gz
    -> Elaborazione: dataset_syzkaller_355_20260409_0145.jsonl.gz
    -> Elaborazione: dataset_syzkaller_356_20260409_0515.jsonl.gz
    -> Elaborazione: dataset_syzkaller_349_20260409_0630.jsonl.gz
    -> Elaborazione: dataset_syzkaller_355_20260408_2215.jsonl.gz
    -> Elaborazione: dataset_syzkaller_354_20260328_0315.jsonl.gz
    -> Elaborazione: dataset_syzkaller_355_20260409_0715.jsonl.gz
    -> Elaborazione: dataset_syzkaller_353_20260327_2330.jsonl.gz
    -> Elaborazione: dataset_syzkaller_355_20260328_0345.jsonl.gz
    -> Elaborazione: dataset_syzkaller_350_20260408_2315.jsonl.gz
    -> Elaborazione: dataset_syzkaller_353_20260409_0545.jsonl.gz
    -> Elaborazione: dataset_syzkaller_353_20260328_0000.jsonl.gz
    -> Elaborazione: dataset_syzkaller_352_20260408_2345.jsonl.gz
    -> Elaborazione: dataset_syzk

In [2]:
import json
from collections import Counter

# Percorso del tuo dataset finale
DATASET_PATH = "/home/stefano-u/fuzzing_lab/shared_corpus/dataset_final_qwen.jsonl"

def analizza_dataset(file_path):
    print(f"[*] Avvio analisi del dataset: {file_path}\n")
    
    totale_righe = 0
    validi = 0
    invalidi = 0
    
    # Raccoglitori di statistiche
    tutte_le_chiavi = set()
    lunghezze_assembly = []
    errori_counter = Counter()
    
    # Per salvare un esempio da guardare
    esempio_valido = None
    esempio_invalido = None

    try:
        with open(file_path, "r", encoding="utf-8") as f:
            for linea in f:
                totale_righe += 1
                data = json.loads(linea)
                
                # Tracciamo tutte le chiavi presenti nel JSON
                tutte_le_chiavi.update(data.keys())
                
                # Analisi validità ed errori
                is_valid = data.get("is_valid", False)
                if is_valid:
                    validi += 1
                    if not esempio_valido:  # <-- CORRETTO QUI
                        esempio_valido = data
                else:
                    invalidi += 1
                    errore = data.get("error_reason_clean", "Errore Sconosciuto")
                    errori_counter[errore] += 1
                    if not esempio_invalido: # <-- CORRETTO QUI
                        esempio_invalido = data
                
                # Cerchiamo di capire come si chiama la chiave dell'Assembly
                # Proviamo i nomi più comuni, modificalo se sai già il nome
                asm_text = data.get("assembly_code", data.get("insns", data.get("prog_text", "")))
                
                # Calcoliamo quante righe di codice (istruzioni) ci sono
                if asm_text:
                    num_istruzioni = len(asm_text.strip().split('\n'))
                    lunghezze_assembly.append(num_istruzioni)

    except FileNotFoundError:
        print("[-] ERRORE: File non trovato. Controlla il percorso DATASET_PATH.")
        return

    # --- STAMPA DEI RISULTATI ---
    print("=== 1. STATISTICHE GENERALI ===")
    print(f"Totale programmi: {totale_righe}")
    print(f"  - Validi:   {validi} ({(validi/totale_righe)*100:.1f}%)")
    print(f"  - Invalidi: {invalidi} ({(invalidi/totale_righe)*100:.1f}%)")
    print("\n=== 2. STRUTTURA DEI DATI (CHIAVI JSON) ===")
    print(f"Le chiavi trovate in ogni riga sono: {list(tutte_le_chiavi)}")
    
    print("\n=== 3. LUNGHEZZA DEL CODICE ASSEMBLY ===")
    if lunghezze_assembly:
        media = sum(lunghezze_assembly) / len(lunghezze_assembly)
        massima = max(lunghezze_assembly)
        minima = min(lunghezze_assembly)
        print(f"Media istruzioni per programma: {media:.1f} righe")
        print(f"Programma più corto: {minima} righe")
        print(f"Programma più lungo: {massima} righe")
        
        # Stimiamo i token (circa 3-4 token per riga di assembly eBPF)
        print(f"-> Consiglio per max_length del tokenizer: {int(massima * 4) + 100}")
    else:
        print("[-] Nessun testo assembly trovato. Controlla il nome della chiave!")

    print("\n=== 4. TOP 5 ERRORI FREQUENTI (Per i programmi invalidi) ===")
    for err, count in errori_counter.most_common(5):
        print(f"  [{count} occorrenze] -> {err}")

    print("\n=== 5. ESEMPIO DI PROGRAMMA VALIDO (Prime 5 righe di Assembly) ===")
    if esempio_valido:
        # Mostriamo il contenuto del campo bytecode o eventuali altre chiavi se assembly non c'è
        print(json.dumps(esempio_valido, indent=2)[:500] + "\n...")

analizza_dataset(DATASET_PATH)

[*] Avvio analisi del dataset: /home/stefano-u/fuzzing_lab/shared_corpus/dataset_final_qwen.jsonl

=== 1. STATISTICHE GENERALI ===
Totale programmi: 27514
  - Validi:   13153 (47.8%)
  - Invalidi: 14361 (52.2%)

=== 2. STRUTTURA DEI DATI (CHIAVI JSON) ===
Le chiavi trovate in ogni riga sono: ['error_reason_clean', 'error_reason', 'bytecode_hex', 'error_line', 'is_valid', 'verifier_log']

=== 3. LUNGHEZZA DEL CODICE ASSEMBLY ===
[-] Nessun testo assembly trovato. Controlla il nome della chiave!

=== 4. TOP 5 ERRORI FREQUENTI (Per i programmi invalidi) ===
  [2000 occorrenze] -> R0 min value is outside of the allowed memory range
  [2000 occorrenze] -> R0 unbounded memory access, make sure to bounds check any such access
  [2000 occorrenze] -> math between map_value pointer and register with unbounded min value is not allowed
  [2000 occorrenze] -> R0 max value is outside of the allowed memory range
  [2000 occorrenze] -> R0 min value is negative, either use unsigned index or do a if (in

In [3]:
import json

DATASET_PATH = "/home/stefano-u/fuzzing_lab/shared_corpus/dataset_final_qwen.jsonl"

chiavi_uniche = set()

print("[*] Estrazione delle chiavi in corso...")

with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    for linea in f:
        try:
            data = json.loads(linea)
            chiavi_uniche.update(data.keys())
        except json.JSONDecodeError:
            continue

print("\n=== CHIAVI TROVATE NEL DATASET ===")
for chiave in sorted(chiavi_uniche):
    print(f"- {chiave}")

[*] Estrazione delle chiavi in corso...

=== CHIAVI TROVATE NEL DATASET ===
- bytecode_hex
- error_line
- error_reason
- error_reason_clean
- is_valid
- verifier_log
